# Phase 06B.01 — Novelty Track Decision & Protocol Lock

Turn slice evidence into one auditable human decision and freeze the common novelty protocol before inspecting novelty validation.

**Immutable gates:** `L32-F1`; 298 frozen validation samples; train-side checkpoint selection only; no public-test access. Missing human/input artifacts produce an explicit status and stop—no synthetic labels or provenance.

## 1. Evidence gate

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
def write_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path
from phase06a_common import sha256_file,sha256_json
from phase06b_common import build_locked_novelty_protocol,load_json,validate_locked_baseline,validate_track_decision
A=ROOT/"outputs/phase06b/taxonomy_slice_analysis"; OUT=ROOT/"outputs/phase06b/novelty_protocol"; OUT.mkdir(parents=True,exist_ok=True)
E=A/"novelty_track_evidence.json"; D=OUT/"novelty_track_decision.json"
baseline=validate_locked_baseline(ROOT)
if not (A/"PHASE06B_00_STATUS.json").is_file() or load_json(A/"PHASE06B_00_STATUS.json").get("status")!="complete":
 raise RuntimeError("Run Phase06B_00 to completion first")
evidence=load_json(E)

## 2. Human lock
Copy the generated template to `novelty_track_decision.json`; fill exactly one supported track, decision maker, and rationale.

In [ ]:
if not D.is_file():
 write_json(OUT/"novelty_track_decision.template.json",{"status":"locked","selected_track":"","decision_by":"","rationale":"",
  "evidence_sha256":sha256_json(evidence),"allowed_tracks":["knowledge_augmented","traffic_temporal_grounding"]})
 write_json(OUT/"PHASE06B_01_STATUS.json",{"status":"awaiting_human_decision","evidence_sha256":sha256_json(evidence)})
 raise RuntimeError("A human novelty-track decision is required")
decision=validate_track_decision(load_json(D),evidence)

## 3. Freeze protocol

In [ ]:
protocol=build_locked_novelty_protocol(decision["selected_track"],baseline_manifest=baseline,
 taxonomy_sha256=sha256_file(ROOT/"outputs/phase06a/question_taxonomy/taxonomy_frozen.csv"),evidence_sha256=sha256_json(evidence))
protocol.update({"decision_sha256":sha256_file(D),"decision_by":decision["decision_by"],"decision_rationale":decision["rationale"]})
write_json(OUT/"novelty_protocol.json",protocol)
write_json(OUT/"PHASE06B_01_STATUS.json",{"status":"complete","selected_track":decision["selected_track"],"protocol_sha256":sha256_json(protocol)})
protocol